# Seventh LAB

### Select the Kernel

Click on "Select Kernel" on the Top Right

Choose "Python Environments..."

Then choose the one that looks like `.venv (Python 3.12.x) .venv/bin/python` - it should be marked as "Recommended" and have a big star next to it.

### Project - Code Generator (SLM version)

In [1]:
# imports
import os
import json
import sqlite3
import gc
import re
import inspect
from threading import Thread
import torch
import time
from dotenv import load_dotenv
from huggingface_hub import login, InferenceClient
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
import shutil
import subprocess
print(shutil.which("gfortran"))

c:\users\karan\gcc\bin\gfortran.EXE


In [2]:
# Check for GPU / CPU
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

GPU: NVIDIA GeForce RTX 3050 Laptop GPU
CUDA: 12.1
Using device: cuda


In [3]:
# Log in to HuggingFace (Llama is a gated model)
load_dotenv(override=True)

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
# Python function
fact = """
import time

def mult(a, times):
    total = 0
    for _ in range(times):
        total += a
    return total

def factorial(n):
    if n <= 1:
        return 1
    return mult(factorial(n - 1), n)

n = 20

start = time.perf_counter()
result = factorial(n)
end = time.perf_counter()

print(f"Digits in {n}!: {len(str(result))}")
print(f"Time taken: {end - start:.6f} seconds")

"""

In [5]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

run_python(fact)

Digits in 20!: 19
Time taken: 0.000016 seconds


### Prompt Definition

In [6]:
def user_prompt(python):
    return f""" 
Port this Python code to fortran with the fastest possible implementation that produces identical output in the least time.
Your response will be written to a file called fact.f90
Respond only with fortran code.
Python code to port:

```python
{python}
```
"""

In [7]:
# Define system message
system_prompt = """
Your task is to convert Python code into high performance fortran code.
Respond only with fortran code. Do not provide any explanation other than occasional comments.
The fortran response needs to produce an identical output in the fastest possible time.
"""

### Model Call

In [12]:
def extract_code(text):
    # Filtering EOD tokens
    m = re.search(r"```(?:fortran|f90)?\s*\n(.*?)(?:```|$)", text, re.DOTALL | re.IGNORECASE)
    code = m.group(1) if m else text
    return code.strip() + "\n"

In [13]:
def model_call(hf_token, model_id, prompt):
    
    # This client sends requests to a remote inference provider.
    client = InferenceClient(
        api_key=hf_token,
        provider="auto"
    )

    # Load model directly
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # Load model directly onto available hardware
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=device,
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
    )

    print("Model loaded successfully!")
    print("Memory footprint:", round(model.get_memory_footprint() / 1024**3, 2), "GB")

    messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt(prompt)}
        ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)

    # Output
    outputs = model.generate(**inputs, max_new_tokens=1024)

    # Extract Response
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:], 
        skip_special_tokens=True)

    # Write output
    with open(f"fact_{model_id[0:4]}.f90", "w", encoding="utf-8") as f:
        f.write(extract_code(response))

    # Free up memory in GPU 
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

In [14]:
# Llama Model
model_id = "meta-llama/Llama-3.2-3B-Instruct"
model_call(hf_token, model_id, fact)


start = time.perf_counter()
c = subprocess.run(["gfortran", f"fact_{model_id[0:4]}.f90", "-o", "fact"], capture_output=True, text=True)
print(c.stderr or "Compiled successfully")
end = time.perf_counter()
print(f"Time taken: {end - start:.6f} seconds")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Model loaded successfully!
Memory footprint: 5.98 GB
fact_meta.f90:10:56:

   10 |         CHARACTER(len=LEN(string)), INTENT(IN) :: string
      |                                                        1
Error: Symbol 'string' at (1) already has basic type of REAL

Time taken: 0.113403 seconds


In [15]:
# Qwen Model
model_id = "Qwen/Qwen2.5-3B-Instruct"
model_call(hf_token, model_id, fact)


start = time.perf_counter()
c = subprocess.run(["gfortran", f"fact_{model_id[0:4]}.f90", "-o", "fact"], capture_output=True, text=True)
print(c.stderr or "Compiled successfully")
end = time.perf_counter()
print(f"Time taken: {end - start:.6f} seconds")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!
Memory footprint: 5.75 GB
fact_Qwen.f90:15:45:

   15 |         res = len(trim(adjustl(merge("", "", int(factorial(n), kind=8)))))
      |                                             1
Error: 'mask' argument of 'merge' intrinsic at (1) must be LOGICAL
fact_Qwen.f90:21:9:

   21 |     use fact_module
      |         1
Fatal Error: Cannot open module file 'fact_module.mod' for reading at (1): No such file or directory
compilation terminated.

Time taken: 0.379325 seconds


In [16]:
# Microsoft Model
model_id = "microsoft/Phi-4-mini-instruct"
model_call(hf_token, model_id, fact)


start = time.perf_counter()
c = subprocess.run(["gfortran", f"fact_{model_id[0:4]}.f90", "-o", "fact"], capture_output=True, text=True)
print(c.stderr or "Compiled successfully")
end = time.perf_counter()
print(f"Time taken: {end - start:.6f} seconds")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!
Memory footprint: 7.15 GB
fact_micr.f90:28:32:

   28 |             fact = n * factorial(n - 1)
      |                                1
Error: Function 'factorial' at (1) cannot be called recursively, as it is not RECURSIVE

Time taken: 0.104770 seconds
